# Detailed CellSpecification Walkthrough

This notebook uses the richer `A123__ANR26650M1-B` library example to show how BattINFO stores
detailed cell-specification metadata beyond the lightweight publication chain.

Focus areas:

- conventional top-level properties
- positive and negative electrode structure
- coating components and mass fractions
- electrolyte and separator detail
- deriving a lightweight public `CellType` from a rich `CellSpecification`


In [ ]:
from pathlib import Path
from pprint import pprint

from battinfo import CellType, load_cell_specification
from battinfo.bundle import Electrode, MaterialComponent

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "assets").exists() and (REPO_ROOT.parent / "assets").exists():
    REPO_ROOT = REPO_ROOT.parent

SPEC_PATH = REPO_ROOT / "assets" / "library" / "cell-types" / "A123__ANR26650M1-B.json"
spec = load_cell_specification(SPEC_PATH)

print("Repo root:", REPO_ROOT)
print("Specification path:", SPEC_PATH)


## Top-Level Summary

In [ ]:
summary = {
    "id": spec.id,
    "manufacturer": spec.manufacturer,
    "model": spec.model,
    "format": spec.format,
    "chemistry": spec.chemistry,
    "positive_electrode_basis": spec.positive_electrode_basis,
    "negative_electrode_basis": spec.negative_electrode_basis,
    "size_code": spec.size_code,
    "property_count": len(spec.properties),
    "example_property_keys": sorted(spec.properties)[:10],
}

pprint(summary)


In [ ]:
key_properties = {
    key: spec.properties[key]
    for key in [
        "nominal_capacity",
        "nominal_voltage",
        "continuous_discharging_current",
        "diameter",
        "height",
        "mass",
    ]
    if key in spec.properties
}

pprint(key_properties)


## Electrode Helpers

In [ ]:
def flatten_components(component_block: dict[str, list[MaterialComponent]]) -> list[dict]:
    rows = []
    for role, items in component_block.items():
        for item in items:
            rows.append(
                {
                    "role": role,
                    "name": item.name,
                    "property": item.property,
                    "comment": item.comment,
                }
            )
    return rows

def summarize_electrode(name: str, electrode: Electrode | None) -> dict:
    if electrode is None:
        return {"name": name, "comment": None, "coating_property": {}, "current_collector": None, "components": []}
    return {
        "name": name,
        "comment": electrode.comment,
        "coating_property": electrode.coating.property if electrode.coating is not None else {},
        "current_collector": electrode.current_collector.model_dump(exclude_none=True) if electrode.current_collector is not None else None,
        "components": flatten_components(electrode.coating.component) if electrode.coating is not None else [],
    }


## Positive Electrode

In [ ]:
positive_summary = summarize_electrode("positive", spec.positive_electrode)
pprint(positive_summary)


## Negative Electrode

In [ ]:
negative_summary = summarize_electrode("negative", spec.negative_electrode)
pprint(negative_summary)


## Electrolyte And Separator

In [ ]:
electrolyte = spec.electrolyte
separator = spec.separator

solvent_mixture = electrolyte.solvent_mixture if electrolyte is not None else None
electrolyte_summary = {
    "family": electrolyte.family if electrolyte is not None else None,
    "solvent_components": [item.model_dump(exclude_none=True) for item in solvent_mixture.component] if solvent_mixture is not None else [],
    "solvent_properties": solvent_mixture.property if solvent_mixture is not None else {},
    "salt": electrolyte.salt.model_dump(exclude_none=True) if electrolyte is not None and electrolyte.salt is not None else None,
    "additives": [item.model_dump(exclude_none=True) for item in electrolyte.additive] if electrolyte is not None else [],
    "bulk_properties": electrolyte.property if electrolyte is not None else {},
    "comment": electrolyte.comment if electrolyte is not None else None,
}

separator_summary = {
    "material": separator.material if separator is not None else None,
    "property": separator.property if separator is not None else {},
    "comment": separator.comment if separator is not None else None,
}

pprint({"electrolyte": electrolyte_summary, "separator": separator_summary})


## Derive The Lightweight Public CellType

The richer `CellSpecification` can be reduced into a public-facing `CellType` for publication workflows.

In [ ]:
cell_type = CellType.from_cell_specification(spec)

pprint(cell_type.model_dump(exclude_none=True))


## Interpretation

- `CellSpecification` keeps the richer engineering detail and provenance.
- `CellType` is the lighter public class-level object used in the publication chain.
- The detailed electrode and materials blocks stay useful for curation, review, and future richer mappings.
